# 09 · The single test run

This is the **only** notebook that reads test-split PM2.5 (forecasting test winter 2024/25) and the network test months (Feb–Mar 2026). It runs **once**, on or after 23 November. Nothing chosen earlier changes: models, configurations, thresholds, features, grids and seeds are those frozen at tag **`pre-test-v2`** (`configs/study_config.yaml`, `outputs/results/forecast_selected.json`, `warning_policy.json`, `network_validation.csv`); their SHA-256 hashes are logged in section 1. All logic lives in `src/uaq/test_run.py`, which only calls the frozen functions.

**The run refuses to start** if the working tree is not clean, HEAD is not the commit tagged `pre-test-v2`, or `outputs/results/test_run_log.json` already exists (no re-runs; the only allowed re-run fixes a bug in evaluation code and is logged with its reason).

## Verdict rules (fixed before any test number is printed)
- **H1** (forecasting): SUPPORTED if the MLP (weather), the neural network selected on validation, has positive MAE skill over persistence on the test smog-season days **and** the 95% interval of MAE(MLP) − MAE(persistence) from a paired 7-day block bootstrap (1,000 resamples, seed 0) lies below 0; otherwise NOT SUPPORTED. The LSTM is reported as secondary.
- **H2** (weather): SUPPORTED if the MLP's test MAE with weather is lower than without, on identical days; all families are reported.
- **H5** (warnings): SUPPORTED if the frozen policy catches more unhealthy days per alert than the single-threshold rule at the matched budget (threshold frozen on validation).
- **H3** (network): SUPPORTED if, for most k, the GA subset has lower test F than random (median), highest-mean and spatial-spread, and test F ≤ greedy.
- **H4** (network, validation result): SUPPORTED if the GA is within 2% of the exhaustive optimum wherever exhaustive search ran; the test F of the GA subset vs the validation-optimal subset is reported.
- **H6** (coal ban): copied from notebook 07 — inconclusive under the placebo rule.

In [ ]:
# Setup: works whether or not you ran `pip install -e .`
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from IPython.display import display, Markdown

from uaq.config import load_config, ROOT
from uaq import test_run as tr
cfg = load_config()
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
%matplotlib inline
print("Project root:", ROOT)

## 1 · Preconditions and log
Checked before any data is read; the log (tag, commit, run time, hashes of the frozen files) is written immediately.

In [ ]:
info = tr.check_preconditions(ROOT)
log = tr.write_log(ROOT, info)
print(json.dumps(log, indent=2))

In [ ]:
res_dir = ROOT / "outputs" / "results"
selected = json.loads((res_dir / "forecast_selected.json").read_text(encoding="utf-8"))
policy = json.loads((res_dir / "warning_policy.json").read_text(encoding="utf-8"))
ban = json.loads((res_dir / "ban_effect.json").read_text(encoding="utf-8"))
feat = pd.read_csv(ROOT / "data" / "processed" / "features_8160.csv", parse_dates=["date", "target_date"])
THR = float(selected["threshold"]["ug_m3"])
print(f"Frozen event threshold {THR} µg/m³; frozen policy {policy['input']} t1={policy['t1']:g} t2={policy['t2']:g}; "
      f"very high {policy['very_high']:.1f} µg/m³; LSTM history {selected['lstm']['history_days']} days")

## 2 · Forecasts on the test winter
Every frozen model is refitted on 2019-07-01 to the end of validation (all months) and predicts the test smog-season days with an observed value. Skill is against persistence on the same days.

In [ ]:
table = tr.forecasts(feat, cfg, selected)
scores = tr.score_forecasts(table, cfg, THR)
cols = ["n", "mae", "rmse", "skill_mae", "precision", "recall", "f1", "missed", "events", "mae_extreme", "bias_extreme", "skill_mae_extreme"]
display(scores[cols].style.format("{:.3f}", subset=[c for c in cols if c not in ("n", "missed", "events")]))

In [ ]:
cmp_fold, cmp_mean = tr.common_days(table, feat, cfg, THR)
display(Markdown("**Common days (days with an LSTM window)**"))
display(cmp_mean[["n", "mae", "rmse", "skill_mae", "precision", "recall", "f1", "missed", "events", "mae_extreme", "bias_extreme"]].round(3))
boot = {m: tr.bootstrap_mae_difference(table, m) for m in ("mlp (weather)", "lstm (weather)")}
display(Markdown("**MAE(model) − MAE(persistence), paired 7-day block bootstrap (1,000 resamples, seed 0)**"))
display(pd.DataFrame(boot).T[["n_days", "mae_difference", "lo", "hi"]].round(2))
ablation = tr.weather_ablation(table)
display(Markdown("**Weather ablation (identical days per family)**")); display(ablation.round(2))
h1, h2 = tr.verdict_h1(scores, boot["mlp (weather)"]), tr.verdict_h2(ablation)
display(Markdown(
    f"**H1: {h1['verdict']}.** MLP (weather) skill vs persistence {h1['mlp_skill_mae']:+.3f}; MAE difference "
    f"{h1['mae_difference']:+.1f} µg/m³ (95% interval {h1['interval'][0]:+.1f} to {h1['interval'][1]:+.1f}). "
    f"Secondary, LSTM (weather): skill {scores.loc['lstm (weather)', 'skill_mae']:+.3f}, MAE difference "
    f"{boot['lstm (weather)']['mae_difference']:+.1f} ({boot['lstm (weather)']['lo']:+.1f} to {boot['lstm (weather)']['hi']:+.1f}).\n\n"
    f"**H2: {h2['verdict']}.** MLP MAE with − without weather: {h2['mlp_mae_difference_weather_minus_none']:+.2f} µg/m³ "
    f"(ridge {ablation.loc['ridge', 'difference']:+.2f}, boosting {ablation.loc['hgb', 'difference']:+.2f}, "
    f"LSTM {ablation.loc['lstm', 'difference']:+.2f})."))

In [ ]:
days = pd.date_range(table.index.min(), table.index.max(), freq="D")
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(days, table["y"].reindex(days), color="k", lw=1.6, label="observed")
ax.plot(days, table["pm_lag0"].reindex(days), ls=":", lw=1.1, label="persistence")
ax.plot(days, table["pred_mlp (weather)"].reindex(days), lw=1.1, label="MLP (weather)")
ax.plot(days, table["pred_lstm (weather)"].reindex(days), lw=1.1, label="LSTM (weather)")
ax.axhline(THR, color="#C44E52", ls="--", lw=1, label=f"event threshold {THR:g} µg/m³")
ax.set_ylabel("daily mean PM2.5 (µg/m³)")
ax.legend(fontsize=8, ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.12))
ax.set_title("Test winter 2024/25: next-day forecasts (gaps = missing observations or no forecast)")
fig.tight_layout(); plt.show()

## 3 · Warning policy on the test winter
The frozen policy and the three single-threshold rules, all on the frozen policy's input forecast.

In [ ]:
w = tr.warning_results(table, feat, cfg, policy)
show = ["alerts", "alerts_l1", "alerts_l2", "high_caught", "high_days", "caught_per_alert", "very_high_caught_l2",
        "very_high_days", "missed", "false_alarms_l1", "false_alarms_l2", "longest_alert_run", "longest_l2_run"]
display(pd.DataFrame(w["cards"]).T[show].style.format({"caught_per_alert": "{:.2f}"}))
h5 = w["h5"]
display(Markdown(f"**H5: {h5['verdict']}.** Unhealthy days caught per alert: policy {h5['policy_caught_per_alert']:.2f} vs "
                 f"{h5['matched_caught_per_alert']:.2f} for the single-threshold rule at the matched (validation) threshold "
                 f"{policy['matched_single_threshold']:.1f} µg/m³."))

In [ ]:
d = w["days"]
cal = pd.date_range(d.index.min(), d.index.max(), freq="D")
cal = cal[cal.month.isin(cfg["smog_months"])]
obs = d["y"].reindex(cal)
band = np.select([obs.isna(), obs > policy["very_high"], obs > cfg["warnings"]["high"]], [0, 3, 2], 1)
fc = d["forecast"].reindex(cal)
rules = ["policy", "single threshold, matched budget", "single threshold", "official line"]
rows = [band] + [np.where(fc.isna(), 0, d[f"level_{r}"].reindex(cal).fillna(0).to_numpy() + 1) for r in rules]
fig, axes = plt.subplots(len(rows), 1, figsize=(12, 3.8), sharex=True)
cm_obs = ListedColormap(["#DDDDDD", "#C6DBEF", "#FD8D3C", "#A50F15"])
cm_lev = ListedColormap(["#DDDDDD", "#FFFFFF", "#FDD0A2", "#D94801"])
for ax, r, cm, lab in zip(axes, rows, [cm_obs] + [cm_lev] * len(rules), ["observed"] + rules):
    ax.imshow(np.asarray(r)[None, :], aspect="auto", cmap=cm, vmin=0, vmax=3, interpolation="nearest")
    ax.set_yticks([0], [lab], fontsize=8)
ticks = [i for i, x in enumerate(cal) if x.day == 1]
axes[-1].set_xticks(ticks, [cal[i].strftime("%b") for i in ticks])
fig.suptitle("Test winter 2024/25: observed band (grey missing, blue below high, orange high, red very high) vs alert level "
             "(white none, light advisory, dark warning, grey no forecast)", fontsize=9)
fig.tight_layout(); plt.show()

## 4 · Network design on the test months (Feb–Mar 2026)
The subsets each method chose on validation (notebook 03) are scored on the test months; the models reconstructing dropped stations are refitted on train + validation, as in notebook 03's sealed cell. No selection is re-run; "random (median)" replays notebook 03's random draws.

In [ ]:
FCOL = "F_test"
net = tr.network_results(cfg, ROOT)
ft = net["table"].pivot(index="k", columns="method", values=FCOL)
display(ft.round(3))
h3, h4 = tr.verdict_h3(net["table"], FCOL), tr.verdict_h4(net["table"], FCOL)
display(Markdown(
    f"**H3: {h3['verdict']}.** The GA subset beats random (median), highest-mean and spatial-spread and matches or beats greedy at "
    f"{len(h3['k_won'])} of {h3['k_total']} values of k ({h3['k_won']}).\n\n"
    f"**H4 (validation): {h4['verdict']}.** GA within 2% of the exhaustive optimum at {h4['k_within_2pct']} of {h4['k_total']} k. "
    f"On the test months the GA subset's F minus the validation-optimal subset's F ranges "
    f"{min(h4['test_F_ga'][k] - h4['test_F_validation_optimum'][k] for k in h4['test_F_ga']):+.3f} to "
    f"{max(h4['test_F_ga'][k] - h4['test_F_validation_optimum'][k] for k in h4['test_F_ga']):+.3f}."))
fig, ax = plt.subplots(figsize=(8, 4))
for m in ft.columns:
    ax.plot(ft.index, ft[m], marker="o", ms=3, lw=2.2 if m == "genetic algorithm" else 1.1, label=m)
ax.set_xlabel("k (stations kept)"); ax.set_ylabel("test F (lower is better)"); ax.legend(fontsize=8)
ax.set_title("Network design on the test months: subsets chosen on validation")
fig.tight_layout(); plt.show()

## 5 · Summary of H1–H6

In [ ]:
summary_rows = [
    {"hypothesis": "H1", "verdict": h1["verdict"], "key number": f"MLP skill {h1['mlp_skill_mae']:+.3f}; MAE diff {h1['mae_difference']:+.1f} "
                                                              f"({h1['interval'][0]:+.1f} to {h1['interval'][1]:+.1f})"},
    {"hypothesis": "H2", "verdict": h2["verdict"], "key number": f"MLP MAE with − without weather {h2['mlp_mae_difference_weather_minus_none']:+.2f}"},
    {"hypothesis": "H3", "verdict": h3["verdict"], "key number": f"GA wins at {len(h3['k_won'])} of {h3['k_total']} k"},
    {"hypothesis": "H4", "verdict": h4["verdict"], "key number": f"GA within 2% of optimum at {h4['k_within_2pct']} of {h4['k_total']} k (validation)"},
    {"hypothesis": "H5", "verdict": h5["verdict"], "key number": f"caught per alert {h5['policy_caught_per_alert']:.2f} vs {h5['matched_caught_per_alert']:.2f} (matched)"},
    {"hypothesis": "H6", "verdict": ban["verdict"], "key number": f"forest placebo {ban['placebo']['rf']['effect']:+.1%} (rule ±15%); "
                                                               f"pooled effect {ban['pooled']['rf']['effect']:+.1%} (notebook 07)"},
]
summary = pd.DataFrame(summary_rows).set_index("hypothesis")
with pd.option_context("display.max_colwidth", 120):
    display(summary)

## 6 · Save
New files only; configs and earlier results are never written. The log was written in section 1.

In [ ]:
net_out = net["table"].assign(stations=net["table"]["stations"].map(lambda s: ", ".join(s) if isinstance(s, list) else ""))
written = tr.save_outputs(ROOT, {
    "test_forecast.csv": table.reset_index(),
    "test_warning.csv": w["days"].reset_index(),
    "test_warning.json": {"policy": {k: policy[k] for k in ("input", "t1", "t2", "very_high", "matched_single_threshold")},
                          "cards": w["cards"], "h5": h5},
    "network_test.csv": net_out,
    "test_summary.json": {"log": log, "threshold": THR,
                          "verdicts": {"H1": h1, "H2": h2, "H3": h3, "H4": h4, "H5": h5,
                                       "H6": {"verdict": ban["verdict"], "source": "notebook 07"}},
                          "forecast_scores": scores[cols].to_dict(orient="index"),
                          "bootstrap": boot, "weather_ablation": ablation.to_dict(orient="index")},
})
for p in written:
    print("Saved", p)